In [11]:
from __future__ import absolute_import
from __future__ import division
from __future__ import print_function

import os
from datetime import datetime, timedelta
import pandas as pd
import math
import numpy as np
import random
from tqdm import trange

from io import BytesIO
from urllib.request import urlopen
from zipfile import ZipFile

from math import sqrt
from pandas import read_csv, DataFrame
from scipy import stats

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import datetime as dt


In [12]:
global save_path
name_sales = 'sales_train_evaluation.csv'
name_calendar = 'calendar.csv'
name_prices = 'sell_prices.csv'
save_name = 'M5'
window_size = 56
stride_size = 7
num_covariates = 6
train_start = '2011-01-29 00:00:00'
train_end = '2016-04-24 00:00:00'
test_start = '2016-03-01 00:00:00' #need additional 7 days as given info
test_end = '2016-05-22 00:00:00'
pred_days = 28
# given_days = 7

In [13]:
save_path = os.path.join('data', save_name)
if not os.path.exists(save_path):
    os.makedirs(save_path)
sales_csv_path = os.path.join(save_path, name_sales)
calendar_csv_path = os.path.join(save_path, name_calendar)
prices_csv_path = os.path.join(save_path, name_prices)

sales = pd.read_csv('../../data/M5/sales_train_evaluation.csv')
sales.name = 'sales'
calendar = pd.read_csv('../../data/M5/calendar.csv')
calendar.name = 'calendar'
prices = pd.read_csv('../../data/M5/sell_prices.csv')
prices.name = 'prices'

#Create date index
date_index = calendar['date']
dates = date_index[0:1941]
dates_list = [dt.datetime.strptime(date, '%Y-%m-%d').date() for date in dates]

# Create a data frame for items sales per day with item ids (with Store Id) as columns names  and dates as the index 
sales['item_store_id'] = sales.apply(lambda x: x['item_id']+'_'+x['store_id'],axis=1)
data_frame = sales.loc[:,'d_1':'d_1941'].T
data_frame.columns = sales['item_store_id'].values

#Set Dates as index 
data_frame = pd.DataFrame(data_frame).set_index([dates_list])
data_frame.index = pd.to_datetime(data_frame.index)
data_frame.head()

data_frame.shape

(1941, 30490)

In [14]:
data_frame.iloc[1913:1914]

,HOBBIES_1_001_CA_1,HOBBIES_1_002_CA_1,HOBBIES_1_003_CA_1,HOBBIES_1_004_CA_1,HOBBIES_1_005_CA_1,HOBBIES_1_006_CA_1,HOBBIES_1_007_CA_1,HOBBIES_1_008_CA_1,HOBBIES_1_009_CA_1,HOBBIES_1_010_CA_1,...,FOODS_3_818_WI_3,FOODS_3_819_WI_3,FOODS_3_820_WI_3,FOODS_3_821_WI_3,FOODS_3_822_WI_3,FOODS_3_823_WI_3,FOODS_3_824_WI_3,FOODS_3_825_WI_3,FOODS_3_826_WI_3,FOODS_3_827_WI_3
2016-04-25,0,0,0,0,1,0,0,19,0,0,...,0,4,2,0,0,0,0,0,1,0


In [15]:
data_frame.iloc[-28:]

,HOBBIES_1_001_CA_1,HOBBIES_1_002_CA_1,HOBBIES_1_003_CA_1,HOBBIES_1_004_CA_1,HOBBIES_1_005_CA_1,HOBBIES_1_006_CA_1,HOBBIES_1_007_CA_1,HOBBIES_1_008_CA_1,HOBBIES_1_009_CA_1,HOBBIES_1_010_CA_1,...,FOODS_3_818_WI_3,FOODS_3_819_WI_3,FOODS_3_820_WI_3,FOODS_3_821_WI_3,FOODS_3_822_WI_3,FOODS_3_823_WI_3,FOODS_3_824_WI_3,FOODS_3_825_WI_3,FOODS_3_826_WI_3,FOODS_3_827_WI_3
2016-04-25,0,0,0,0,1,0,0,19,0,0,...,0,4,2,0,0,0,0,0,1,0
2016-04-26,0,1,0,0,0,0,0,3,2,2,...,4,2,2,0,2,0,1,0,3,0
2016-04-27,0,0,1,1,2,1,0,2,6,1,...,0,0,1,0,2,0,1,1,0,0
2016-04-28,2,0,1,2,3,0,0,8,0,0,...,1,0,1,3,2,2,1,1,1,0
2016-04-29,0,0,0,4,1,0,1,8,0,0,...,2,1,1,1,2,2,0,0,2,0
2016-04-30,3,0,2,1,0,2,0,23,0,0,...,1,2,1,2,5,0,0,2,1,1
2016-05-01,5,0,1,6,3,4,1,26,0,0,...,3,6,0,0,5,0,0,1,0,1
2016-05-02,0,0,0,4,2,1,0,9,0,0,...,3,2,1,0,4,0,0,1,2,1
2016-05-03,0,0,0,0,3,0,1,4,0,0,...,4,2,1,1,2,2,1,0,1,2
2016-05-04,1,1,0,0,1,0,0,8,0,0,...,4,0,1,1,3,0,0,0,1,0


In [16]:
data_frame.iloc[1801:1913]

,HOBBIES_1_001_CA_1,HOBBIES_1_002_CA_1,HOBBIES_1_003_CA_1,HOBBIES_1_004_CA_1,HOBBIES_1_005_CA_1,HOBBIES_1_006_CA_1,HOBBIES_1_007_CA_1,HOBBIES_1_008_CA_1,HOBBIES_1_009_CA_1,HOBBIES_1_010_CA_1,...,FOODS_3_818_WI_3,FOODS_3_819_WI_3,FOODS_3_820_WI_3,FOODS_3_821_WI_3,FOODS_3_822_WI_3,FOODS_3_823_WI_3,FOODS_3_824_WI_3,FOODS_3_825_WI_3,FOODS_3_826_WI_3,FOODS_3_827_WI_3
2016-01-04,2,0,0,0,0,0,0,2,0,2,...,5,2,2,0,1,4,0,0,0,1
2016-01-05,0,0,0,1,0,0,0,1,0,1,...,1,3,1,3,2,0,0,2,0,1
2016-01-06,1,0,0,1,0,0,2,2,0,0,...,2,4,1,1,1,6,0,2,0,0
2016-01-07,0,0,0,0,2,0,2,75,0,0,...,1,4,3,0,2,1,0,0,3,2
2016-01-08,1,0,0,2,0,0,0,8,0,0,...,3,2,1,2,3,0,0,3,3,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2016-04-20,1,1,1,0,1,0,1,4,0,0,...,4,1,1,0,0,0,0,1,1,0
2016-04-21,3,0,0,1,2,0,0,6,0,0,...,2,3,3,0,2,1,0,0,0,0
2016-04-22,0,0,1,3,2,2,0,3,0,2,...,0,1,6,0,3,0,0,0,3,0
2016-04-23,1,0,1,7,2,0,1,2,0,0,...,3,0,0,4,2,0,1,1,1,0


In [17]:
def gen_m5_covariates(calendar_df):
    # Assuming calendar_df is the calendar DataFrame and sales_df is the sales data DataFrame
    num_covariates = 6  # Example: day of week, month, SNAP_CA, SNAP_TX, SNAP_WI, event
    times = pd.to_datetime(calendar_df['date'])
    covariates = np.zeros((len(times), num_covariates))
    # Day of week and month as numerical values
    covariates[:, 0] = times.dt.dayofweek
    covariates[:, 1] = times.dt.month
    # SNAP days for CA, TX, and WI
    covariates[:, 2] = calendar_df['snap_CA']
    covariates[:, 3] = calendar_df['snap_TX']
    covariates[:, 4] = calendar_df['snap_WI']
    # Event (simple binary indicator for this example)
    covariates[:, 5] = calendar_df[['event_name_1', 'event_name_2']].notnull().any(axis=1).astype(int)
    # Standardize covariates (optional depending on model requirements)
    for i in range(num_covariates):
        covariates[:, i] = (covariates[:, i] - covariates[:, i].mean()) / covariates[:, i].std()
    return covariates

In [18]:
covariates = gen_m5_covariates(calendar)
train_data = data_frame[train_start:train_end].values
test_data = data_frame[test_start:test_end].values
data_start = (train_data!=0).argmax(axis=0) #find first nonzero value in each time series
total_time = data_frame.shape[0] #1941 (days)
num_series = data_frame.shape[1] #30490 (item_id)
covariates = covariates[:1941,:]

In [19]:
def prep_data(data, covariates, data_start, train = True):

    print("train: ", train)
    time_len = data.shape[0] # time_len = 1913
    
    print("time_len: ", time_len)
    input_size = window_size-stride_size # input_size = 192-24 = 168
    
    #每個item有幾個window
    windows_per_series = np.full((num_series), (time_len-input_size) // stride_size) #(193-168)//24
    print("windows pre: ", windows_per_series.shape)

    if train: windows_per_series -= (data_start+stride_size-1) // stride_size
    print("data_start: ", data_start.shape)
    print(data_start)
    print("windows: ", windows_per_series.shape)
    print(windows_per_series)

    # bug is from here!!!!!!
    ##################################################
    # the original way to count windows_per_series and sum(count windows_per_series[series]) is inconsistant
    # so i recalculate it

    # total_windows = np.sum(windows_per_series) // original one

    # new one
    total_windows = 0
    for series in trange(num_series):
        for i in range(windows_per_series[series]):
            total_windows += 1
    print("total_windows: ", total_windows)
    ##################################################

    x_input = np.zeros((total_windows, window_size, 1 + num_covariates + 1), dtype='float32')
    label = np.zeros((total_windows, window_size), dtype='float32')
    v_input = np.zeros((total_windows, 2), dtype='float32')
    print("x_input: ", x_input.shape)
    print("label: ", label.shape)
    print("v_input: ", v_input.shape)
    print(num_series)

    #cov = 3: ground truth + age + day_of_week + hour_of_day + num_series
    #cov = 4: ground truth + age + day_of_week + hour_of_day + month_of_year + num_series
    
    
    # k = 0
    # m = 0
    # print(windows_per_series[0])
    # print(windows_per_series[1])
    # print(windows_per_series[2])
    # print(windows_per_series[3])
    # for series in trange(num_series):
    #     # print(sum((windows_per_series[series])))
    #     m += windows_per_series[series]
    #     for i in range(windows_per_series[series]):
    #         k += 1
    # print('m = ', m)
    # print('k = ', k)
    
    count = 0
    if not train:
        covariates = covariates[-time_len:]
    for series in trange(num_series):
        cov_age = stats.zscore(np.arange(total_time-data_start[series]))
        if train:
            covariates[data_start[series]:time_len, 0] = cov_age[:time_len-data_start[series]]
        else:
            covariates[:, 0] = cov_age[-time_len:]
            
        for i in range(windows_per_series[series]):
            if train:
                window_start = stride_size*i+data_start[series]
            else:
                window_start = stride_size*i
            window_end = window_start+window_size
            
            # if(count > 100000):
            #     print("x: ", x_input[count, 1:, 0].shape)
            #     print("window start: ", window_start)
            #     print("window end: ", window_end)
            #     print("data: ", data.shape)
            #     print("d: ", data[window_start:window_end-1, series].shape)
            
            try:
                x_input[count, 1:, 0] = data[window_start:window_end-1, series]
            except:
                print(count, window_start, window_end, series)
            x_input[count, :, 1:1+num_covariates] = covariates[window_start:window_end, :]
            x_input[count, :, -1] = series
            label[count, :] = data[window_start:window_end, series]
            nonzero_sum = (x_input[count, 1:input_size, 0]!=0).sum()
            if nonzero_sum == 0:
                v_input[count, 0] = 0
            else:
                v_input[count, 0] = np.true_divide(x_input[count, 1:input_size, 0].sum(),nonzero_sum)+1
                x_input[count, :, 0] = x_input[count, :, 0]/v_input[count, 0]
                if train:
                    label[count, :] = label[count, :]/v_input[count, 0]
            count += 1

    prefix = os.path.join(save_path, 'train_' if train else 'test_')
    np.save(prefix+'data_'+ save_name, x_input)
    np.save(prefix+'v_'+ save_name, v_input)
    np.save(prefix+'label_'+ save_name, label)

In [20]:
prep_data(train_data, covariates, data_start)
prep_data(test_data, covariates, data_start, train=False)

train:  True
time_len:  1913
windows pre:  (30490,)
data_start:  (30490,)
[ 901  143 1105 ...    1  939 1130]
windows:  (30490,)
[137 245 108 ... 265 131 104]


100%|██████████| 30490/30490 [00:00<00:00, 213683.30it/s]


total_windows:  6330557
x_input:  (6330557, 56, 8)
label:  (6330557, 56)
v_input:  (6330557, 2)
30490


100%|██████████| 30490/30490 [00:43<00:00, 707.49it/s]


train:  False
time_len:  83
windows pre:  (30490,)
data_start:  (30490,)
[ 901  143 1105 ...    1  939 1130]
windows:  (30490,)
[4 4 4 ... 4 4 4]


100%|██████████| 30490/30490 [00:00<00:00, 2936494.35it/s]


total_windows:  121960
x_input:  (121960, 56, 8)
label:  (121960, 56)
v_input:  (121960, 2)
30490


100%|██████████| 30490/30490 [00:01<00:00, 17801.84it/s]
